# tile-based post processing
1) query and cp files to set tile/year/.tif structure in public-s3
2) 

The outputs are I accidentally hardcoded the epsg to 32618 so MGRS tile 31PGR is in the wrong epsg projection, what projection should they be in and can you help reproject them?

In [1]:
import os
import shutil
from pathlib import Path
from collections import defaultdict

In [2]:
def flatten_phenometrics_outputs(
    results_dir:  str | Path,
    flat_dir:    str | Path,
    tile:        str,
    prefix:      str = "",
    years:       list[int] = [],
    dry_run:     bool = True,
    on_conflict: str  = "skip",   # "skip" | "overwrite" | "error"
    phenometrics_only: bool = True,
) -> dict[str, int]:
    """
    Flatten deep phenometrics output prefix into a single tile directory.
    Deep structure  : <output_dir>/<tile>_<year>/*/*/*/*/*/*/*/<tile>/<year>/*.tif
    Flat target     : <flat_dir>/<tile>/<filename>.tif
                      (year is already embedded in every filename)
    """
    results_dir = Path(results_dir)
    flat_dir   = Path(flat_dir)
    tile_flat  = flat_dir / tile

    if not dry_run:
        tile_flat.mkdir(parents=True, exist_ok=True)

    stats       = defaultdict(int)
    seen_names  = {} 

    for year in years:
        search_root = results_dir / f"{prefix}{tile}-{year}"

        if not search_root.exists():
            print(f"  ! Source dir not found, skipping: {search_root}")
            stats["missing_source"] += 1
            continue
        if phenometrics_only:
            tif_files = sorted(search_root.rglob(f"*{year}.tif"))
        else:
            tif_files = sorted(search_root.rglob(f"*.tif"))

        if not tif_files:
            print(f"  ! No .tif files found under {search_root}")
            continue

        print(f"\n  {tile}_{year}: {len(tif_files)} files found")

        for src in tif_files:
            fname = src.name
            dst   = tile_flat / fname

            # ── Conflict check ──────────────────────────────────────────
            if fname in seen_names:
                print(
                    f"  [DUPLICATE] {fname}\n"
                    f"    first : {seen_names[fname]}\n"
                    f"    second: {src}"
                )
                stats["duplicates"] += 1

            seen_names[fname] = src

            if dst.exists() and not dry_run:
                if on_conflict == "skip":
                    print(f"  [SKIP]  {fname} (already exists)")
                    stats["skipped"] += 1
                    continue
                elif on_conflict == "error":
                    raise FileExistsError(f"Conflict: {dst} already exists")
                #else overwrite — fall through to copy

            # ── Copy ────────────────────────────────────────────────────
            if dry_run:
                print(f"  [DRY]  {src.relative_to(results_dir)}")
                print(f"      :  {dst.relative_to(flat_dir)}")
                stats["would_copy"] += 1
            else:
                try:
                    shutil.copy2(src, dst)
                    print(f"  [OK]   {fname}")
                    stats["copied"] += 1
                except Exception as e:
                    print(f"  [ERR]  {fname}: {e}")
                    stats["errors"] += 1

    return dict(stats)

In [3]:
# ── Confirm and execute ──────────────────────────────────────────────
# if confirm.strip().lower() == "y":
# for t in tiles:
#     print(f"\n=== COPYING {t} ===")
#     stats = flatten_phenometrics_outputs(
#         results_dir  = f"{results_dir}",
#         flat_dir    = FLAT_DIR,
#         prefix      = prefix,
#         tile        = t,
#         years       = years,
#         dry_run     = False,
#         on_conflict = "skip",
#         phenometrics_only = True,
#     )
#     print(f"\nFinal summary: {stats}")

In [4]:
from pathlib import Path
from collections import defaultdict
import os
import shutil


def find_tile_year_dirs(search_root: Path, tile: str, year: int, max_depth: int = 20):
    """
    Efficiently find directories ending in /<tile>/<year>/ without collecting
    every .tif in the entire deep tree.

    Example target:
        .../18SUJ/2024/
    """

    search_root = Path(search_root)
    year_str = str(year)

    prune_names = {
        ".git",
        ".venv",
        "__pycache__",
        "temp_downloads",
        "_hls_downloads",
        "logs",
        "log",
        "tmp",
        "cache",
    }

    root_depth = len(search_root.parts)

    for dirpath, dirnames, filenames in os.walk(search_root, topdown=True, followlinks=False):
        current = Path(dirpath)

        # Prevent runaway scanning if the tree is unexpectedly deep
        depth = len(current.parts) - root_depth
        if depth > max_depth:
            dirnames[:] = []
            continue

        # Prune known irrelevant directories
        dirnames[:] = [
            d for d in dirnames
            if d not in prune_names and not d.startswith(".")
        ]

        # Target terminal directory is .../<tile>/<year>
        if current.name == year_str and current.parent.name == tile:
            yield current

            # Do not descend below terminal output directory
            dirnames[:] = []


def is_phenometric_tif(path: Path, year: int) -> bool:
    """
    Filter to phenometric GeoTIFFs.

    Adjust this if your phenometric filenames have a stricter naming convention.
    """

    name = path.name

    if path.suffix.lower() not in {".tif", ".tiff"}:
        return False

    # Exclude non-phenometric outputs
    exclude_tokens = [
        "EVI2",
        "PreMaskCount",
        "PostMaskCount",
        "ViablePct",
        "Fmask",
    ]

    if any(tok in name for tok in exclude_tokens):
        return False

    # Preserve your old behavior approximately:
    # old code used rglob(f"*{year}.tif")
    if not name.endswith(f"{year}.tif"):
        return False

    return True


def flatten_phenometrics_outputs_fast(
    results_dir: str | Path,
    flat_dir: str | Path,
    tile: str,
    prefix: str = "",
    years: list[int] | None = None,
    dry_run: bool = True,
    on_conflict: str = "skip",
    phenometrics_only: bool = True,
    verbose: bool = False,
    copy_metadata: bool = False,
    max_depth: int = 20,
    skip_tile_if_populated: bool = True,   
    remove_empty_tile_dir: bool = True,  
) -> dict[str, int]:

    if years is None:
        years = []

    if on_conflict not in {"skip", "overwrite", "error"}:
        raise ValueError("on_conflict must be one of: skip, overwrite, error")

    results_dir = Path(results_dir)
    flat_dir = Path(flat_dir)
    tile_flat = flat_dir / tile

    # ── NEW: skip entire tile if output dir already has .tif files ──────────
    if skip_tile_if_populated and not dry_run and tile_flat.exists():
        existing_tifs = list(tile_flat.glob("*.tif"))
        if existing_tifs:
            print(f"  [SKIP TILE] {tile_flat} already contains "
                  f"{len(existing_tifs)} .tif file(s). Skipping.")
            return {"tile_skipped_populated": 1}
    # ────────────────────────────────────────────────────────────────────────

    if not dry_run:
        tile_flat.mkdir(parents=True, exist_ok=True)

    stats = defaultdict(int)
    seen_names = {}
    copy_func = shutil.copy2 if copy_metadata else shutil.copyfile

    for year in years:
        search_root = results_dir / f"{prefix}{tile}-{year}"

        if not search_root.exists():
            print(f"  ! Source dir not found, skipping: {search_root}")
            stats["missing_source"] += 1
            continue

        terminal_dirs = list(
            find_tile_year_dirs(
                search_root=search_root,
                tile=tile,
                year=year,
                max_depth=max_depth,
            )
        )

        if not terminal_dirs:
            print(f"  ! No terminal /{tile}/{year}/ dirs found under {search_root}")
            stats["missing_terminal_dir"] += 1
            continue

        tif_files = []

        for terminal_dir in terminal_dirs:
            for p in terminal_dir.iterdir():
                if not p.is_file():
                    continue
                if phenometrics_only:
                    if is_phenometric_tif(p, year):
                        tif_files.append(p)
                else:
                    if p.suffix.lower() in {".tif", ".tiff"}:
                        tif_files.append(p)

        tif_files = sorted(set(tif_files))

        if not tif_files:
            print(f"  ! No matching .tif files found for {tile} {year}")
            stats["no_files"] += 1
            continue

        print(f"  {tile}_{year}: {len(tif_files)} files found in "
              f"{len(terminal_dirs)} terminal dirs")

        for src in tif_files:
            fname = src.name
            dst = tile_flat / fname

            if fname in seen_names:
                if verbose:
                    print(
                        f"  [DUPLICATE] {fname}\n"
                        f"    first : {seen_names[fname]}\n"
                        f"    second: {src}"
                    )
                stats["duplicates"] += 1

            seen_names[fname] = src

            if dst.exists() and not dry_run:
                if on_conflict == "skip":
                    if verbose:
                        print(f"  [SKIP] {fname} already exists")
                    stats["skipped"] += 1
                    continue
                if on_conflict == "error":
                    raise FileExistsError(f"Conflict: {dst} already exists")

            if dry_run:
                if verbose:
                    print(f"  [DRY] {src} -> {dst}")
                stats["would_copy"] += 1
            else:
                try:
                    copy_func(src, dst)
                    if verbose:
                        print(f"  [OK] {fname}")
                    stats["copied"] += 1
                except Exception as e:
                    print(f"  [ERR] {fname}: {e}")
                    stats["errors"] += 1

    # ── NEW: remove tile dir if nothing was actually copied ──────────────────
    if remove_empty_tile_dir and not dry_run and tile_flat.exists():
        copied_count = stats.get("copied", 0)
        if copied_count == 0:
            try:
                shutil.rmtree(tile_flat)
                print(f"  [CLEANUP] Removed empty tile dir: {tile_flat}")
                stats["empty_dir_removed"] = 1
            except Exception as e:
                print(f"  [CLEANUP ERR] Could not remove {tile_flat}: {e}")
    # ────────────────────────────────────────────────────────────────────────

    return dict(stats)

In [5]:
results_dir =  "/home/jovyan/my-private-bucket/dps_output/hls-phenometrics/main/"
FLAT_DIR   =  "/home/jovyan/my-public-bucket/HLS_phenometrics/"

In [6]:
# prefix = "DMV-"
# dmv_tiles = [
#     "17SPD", "17SQD", "18STJ", "18SUJ", "18SVJ",
#     "17SPC", "17SQC", "18STH", "18SUH", "18SVH",
#     "17SPB", "17SQB", "18STG", "18SUG",
# ]
# years      = ["2024", "2025"]# list(range(2015, 2025+1))
# tiles       = dmv_tiles

In [7]:
# prefix = "AK-"
# ak_mgrs_tiles = [
#     "03VWG", "03VWH", "03VWJ", "03VWK", "03VWL",
#     "03VXF", "03VXG", "03VXH", "03VXJ", "03VXK", "03VXL",
#     "03WVM", "03WVN", "03WVP", "03WVR",
#     "03WWM", "03WWN", "03WWP", "03WWQ", "03WWR", "03WWS", "03WWT",
#     "03WXM", "03WXN", "03WXP", "03WXQ", "03WXR", "03WXS",
#     "04VCL", "04VCM", "04VCN", "04VCP", "04VCQ", "04VCR",
#     "04VDK", "04VDL", "04VDM", "04VDN", "04VDP", "04VDQ", "04VDR",
#     "04VEK", "04VEL", "04VEM", "04VEN", "04VEP", "04VEQ", "04VER",
#     "04VFK", "04VFL", "04VFM", "04VFN", "04VFP", "04VFQ", "04VFR",
#     "04WDA", "04WDB", "04WDC", "04WDD", "04WDR", "04WDS", "04WDT", "04WDU", "04WDV",
#     "04WEA", "04WEB", "04WEC", "04WED", "04WES", "04WET", "04WEU", "04WEV",
#     "04WFA", "04WFB", "04WFR", "04WFS", "04WFT", "04WFU", "04WFV",
#     "05VLE", "05VLF", "05VLG", "05VLH", "05VLJ", "05VLK", "05VLL",
#     "05VME", "05VMF", "05VMG", "05VMH", "05VMJ", "05VMK", "05VML",
#     "05VNE", "05VNF", "05VNG", "05VNH", "05VNJ", "05VNK", "05VNL",
#     "05VPF", "05VPG", "05VPH", "05VPJ", "05VPK", "05VPL",
#     "05WLL", "05WLM", "05WLN", "05WLP", "05WLQ", "05WLR", "05WLS", "05WLT", "05WLU",
#     "05WML", "05WMM", "05WMN", "05WMP", "05WMQ", "05WMR", "05WMS", "05WMT", "05WMU",
#     "05WNL", "05WNM", "05WNN", "05WNP", "05WNQ", "05WNR", "05WNS", "05WNT", "05WNU",
#     "05WPL", "05WPM", "05WPN", "05WPP", "05WPQ", "05WPR", "05WPS",
#     "06VUL", "06VUM", "06VUN", "06VUP", "06VUQ", "06VUR",
#     "06VVL", "06VVM", "06VVN", "06VVP", "06VVQ", "06VVR",
#     "06VWM", "06VWN", "06VWP", "06VWQ", "06VWR",
#     "06VXM", "06VXN", "06VXP", "06VXQ", "06VXR",
#     "06WUA", "06WUB", "06WUC", "06WUD", "06WUR", "06WUS", "06WUT", "06WUU", "06WUV",
#     "06WVA", "06WVB", "06WVC", "06WVD", "06WVR", "06WVS", "06WVT", "06WVU", "06WVV",
#     "06WWA", "06WWB", "06WWC", "06WWD", "06WWR", "06WWS", "06WWT", "06WWU", "06WWV",
#     "06WXA", "06WXB", "06WXR", "06WXS", "06WXT", "06WXU", "06WXV",
#     "07VCG", "07VCH", "07VCJ", "07VCK", "07VCL",
#     "07VDG", "07VDH", "07VDJ", "07VDK", "07VDL",
#     "07WCL", "07WCM", "07WCN", "07WCP", "07WCQ", "07WCR", "07WCS", "07WCT", "07WCU",
#     "07WDL", "07WDM", "07WDN", "07WDP", "07WDQ", "07WDR", "07WDS", "07WDT", "07WDU",
# ]
# years = [2024]# list(range(2015, 2025+1))
# tiles = ak_mgrs_tiles

In [8]:
# ak2 = ['03VWK', '03WVN', '03WXQ', '04VCP', '04VDR', '04VEP', '04VFK', '04VFL', '04VFM', '04VFN', 
#        '04WDC', '04WDR', '04WEA', '04WEB', '04WFA', '04WFR', '04WFV', '05VLE', '05VLK', '05VMH', 
#        '05VMK', '05VNJ', '05VPJ', '05WLL', '05WLM', '05WLN', '05WLP', '05WLQ', '05WLR', '05WLS', 
#        '05WLT', '05WLU', '05WML', '05WNL', '05WNM', '05WNQ', '05WPL', '05WPQ', '06VVL', '06VWQ', 
#        '06VWR', '06VXQ', '06WUA', '06WUB', '06WUC', '06WUD', '06WUR', '06WUS', '06WUT', '06WUU', 
#        '06WUV', '06WVR', '06WWR', '06WWU', '06WXA', '06WXR', '06WXU', '07VDH', '07VDL', '07WCL', 
#        '07WCM', '07WCN', '07WCP', '07WCQ', '07WCR', '07WCS', '07WCT', '07WCU', '07WDL', '07WDN', 
#        '07WDP', '07WDQ', '07WDU']
# years=[2024]
# tiles=ak2
# prefix="AK2-"

In [9]:
# prefix = "TS-"
# tiles = ["18SUJ"]
# years = list(range(2015, 2026))
# years

In [10]:
# tiles = ['12TYP', '12TYQ', '13TCJ', '13TCK']
# years = list(range(2015, 2025+1))
# prefix = "CCM-"

In [11]:
# tiles = ['12SWE','12SWF','12SVF', '12SVE']
# years = list(range(2015, 2025+1))
# prefix = "Hopi-"

In [15]:
tiles = ['04VFN', '04VFP', '04VFQ', '04VFR', '04WEB', '04WEC', '04WED', '04WFA', '04WFS', 
 '04WFT', '04WFU', '04WFV', '05VLG', '05VLH', '05VLJ', '05VLK', '05VLL']
years = list(range(2015, 2025+1))
prefix="AGU-"

In [13]:
# my-private-bucket/dps_output/hls-phenometrics/main/Test-18SUJ-2025/2026/07/28/17/51/30/965201
# Test 18SUJ 2013-2025
# AGU 12TYP 2013-2025
# AGU ['04VFN', '04VFP', '04VFQ', '04VFR', '04WEB', '04WEC', '04WED', '04WFA', '04WFS', 
#          '04WFT', '04WFU', '04WFV', '05VLG', '05VLH', '05VLJ', '05VLK', '05VLL'] 2015-2025
tiles = ['12TYP']
years = list(range(2013, 2025+1))
prefix = "Test-"

In [16]:
import gc
import json

CHECKPOINT_FILE = Path("flatten_checkpoint.json")

def load_completed() -> set:
    if CHECKPOINT_FILE.exists():
        return set(json.loads(CHECKPOINT_FILE.read_text()))
    return set()

def mark_done(tile: str, done: set):
    done.add(tile)
    CHECKPOINT_FILE.write_text(json.dumps(sorted(done)))

completed = load_completed()
all_stats = {}

for t in tiles:
    # if t in completed:
    #     print(f"  [ALREADY DONE] {t}")
    #     continue

    print(f"\n=== COPYING {t} ===")
    try:
        stats = flatten_phenometrics_outputs_fast(
            results_dir=results_dir,
            flat_dir=FLAT_DIR,
            prefix=prefix,
            tile=t,
            years=years,
            dry_run=False,
            on_conflict="skip",
            phenometrics_only=True,
            verbose=False,
            copy_metadata=False,
            max_depth=20,
            skip_tile_if_populated=False,
            remove_empty_tile_dir=False,
        )
    except Exception as e:
        print(f"  [FATAL ERROR] {t}: {e}")
        continue   

    all_stats[t] = stats
    print(f"Final summary for {t}: {stats}")

    mark_done(t, completed)
    gc.collect()


=== COPYING 04VFN ===
  04VFN_2015: 40 files found in 2 terminal dirs
  04VFN_2016: 20 files found in 1 terminal dirs
  04VFN_2017: 20 files found in 1 terminal dirs
  04VFN_2018: 20 files found in 1 terminal dirs
  04VFN_2019: 20 files found in 1 terminal dirs
  ! No terminal /04VFN/2020/ dirs found under /home/jovyan/my-private-bucket/dps_output/hls-phenometrics/main/AGU-04VFN-2020
  04VFN_2021: 20 files found in 1 terminal dirs
  ! No terminal /04VFN/2022/ dirs found under /home/jovyan/my-private-bucket/dps_output/hls-phenometrics/main/AGU-04VFN-2022
  04VFN_2023: 20 files found in 1 terminal dirs
  04VFN_2024: 20 files found in 1 terminal dirs
  04VFN_2025: 20 files found in 1 terminal dirs
Final summary for 04VFN: {'skipped': 200, 'duplicates': 20, 'missing_terminal_dir': 2}

=== COPYING 04VFP ===
  04VFP_2015: 20 files found in 1 terminal dirs
  04VFP_2016: 20 files found in 1 terminal dirs
  04VFP_2017: 20 files found in 1 terminal dirs
  04VFP_2018: 20 files found in 1 termina

In [17]:
all_stats

{'04VFN': {'skipped': 200, 'duplicates': 20, 'missing_terminal_dir': 2},
 '04VFP': {'skipped': 240, 'duplicates': 20},
 '04VFQ': {'skipped': 180, 'missing_terminal_dir': 1, 'missing_source': 1},
 '04VFR': {'skipped': 180, 'missing_source': 2},
 '04WEB': {'skipped': 256, 'duplicates': 36},
 '04WEC': {'missing_source': 2,
  'skipped': 180,
  'missing_terminal_dir': 1,
  'duplicates': 20},
 '04WED': {'skipped': 200,
  'duplicates': 20,
  'missing_terminal_dir': 1,
  'missing_source': 1},
 '04WFA': {'skipped': 179,
  'missing_terminal_dir': 2,
  'duplicates': 19,
  'missing_source': 1},
 '04WFS': {'skipped': 220, 'duplicates': 20, 'missing_source': 1},
 '04WFT': {'skipped': 200, 'duplicates': 40, 'missing_source': 3},
 '04WFU': {'skipped': 160, 'missing_terminal_dir': 1, 'missing_source': 2},
 '04WFV': {'skipped': 148, 'missing_terminal_dir': 2, 'missing_source': 1},
 '05VLG': {'skipped': 241, 'missing_source': 1, 'duplicates': 41},
 '05VLH': {'skipped': 220, 'duplicates': 40, 'missing_ter

In [9]:
# all_stats = {}

# for t in tiles:
#     print(f"\n=== COPYING {t} ===")
#     stats = flatten_phenometrics_outputsa_fast(
#         results_dir=results_dir,
#         flat_dir=FLAT_DIR,
#         prefix=prefix,
#         tile=t,
#         years=years,
#         dry_run=False,
#         on_conflict="skip",
#         phenometrics_only=True,
#         verbose=False,          # important for speed / stable logs
#         copy_metadata=False,    # faster than shutil.copy2
#         max_depth=20,
#     )
    
#     all_stats[t] = stats
#     gc.collect()
#     print(f"Final summary for {t}: {stats}")

# TEMP: retag wrong epsg


In [21]:
from pathlib import Path
import rasterio as rio
from rasterio.crs import CRS

SRC_EPSG = 32618
DST_EPSG = 32631
TILE_DIR = Path("/projects/my-public-bucket/HLS_phenometrics/31PGR")

In [24]:
WRONG_CRS = CRS.from_epsg(32618)
RIGHT_CRS = CRS.from_epsg(32631)

ok      = []
retagged = []
unknown = []

for tif in sorted(TILE_DIR.glob("*.tif")):
    with rio.open(tif) as ds:
        current_crs = ds.crs

    if current_crs == RIGHT_CRS:
        ok.append(tif.name)

    elif current_crs == WRONG_CRS:
        with rio.open(tif, "r+") as ds:
            ds.crs = RIGHT_CRS
        retagged.append(tif.name)
        print(f"  [RETAGGED] {tif.name}")

    else:
        unknown.append((tif.name, current_crs))
        print(f"  [UNKNOWN]  {tif.name}  →  {current_crs}")

print(f"\nSummary:")
print(f"  Already correct : {len(ok)}")
print(f"  Retagged        : {len(retagged)}")
print(f"  Unknown CRS     : {len(unknown)}")
if unknown:
    print(f"\n  Unknown files:")
    for name, crs in unknown:
        print(f"    {name}  :  {crs}")

  [RETAGGED] amplitude_2016.tif
  [RETAGGED] amplitude_2017.tif
  [RETAGGED] amplitude_2022.tif
  [RETAGGED] auc_full_2015.tif
  [RETAGGED] auc_full_2016.tif
  [RETAGGED] auc_full_2017.tif
  [RETAGGED] auc_full_2022.tif
  [RETAGGED] auc_net_2015.tif
  [RETAGGED] auc_net_2016.tif
  [RETAGGED] auc_net_2017.tif
  [RETAGGED] auc_net_2022.tif
  [RETAGGED] dormancy_doy_2015.tif
  [RETAGGED] dormancy_doy_2016.tif
  [RETAGGED] dormancy_doy_2017.tif
  [RETAGGED] dormancy_doy_2022.tif
  [RETAGGED] dormancy_evi_2015.tif
  [RETAGGED] dormancy_evi_2016.tif
  [RETAGGED] dormancy_evi_2017.tif
  [RETAGGED] dormancy_evi_2022.tif
  [RETAGGED] greenup_doy_2015.tif
  [RETAGGED] greenup_doy_2016.tif
  [RETAGGED] greenup_doy_2017.tif
  [RETAGGED] greenup_doy_2022.tif
  [RETAGGED] greenup_evi_2015.tif
  [RETAGGED] greenup_evi_2016.tif
  [RETAGGED] greenup_evi_2017.tif
  [RETAGGED] greenup_evi_2022.tif
  [RETAGGED] greenup_rate_2015.tif
  [RETAGGED] greenup_rate_2016.tif
  [RETAGGED] greenup_rate_2017.tif
  [